# 12. Backtesting and Model Validation

Part of the `nuclearff` [User Tutorial](https://nolmacdonald.github.io/nuclearff/tutorial/index.html) notebook series -- every notebook shares one running `./demo` directory and depends on the ones before it having already run. See [README.md](README.md) for the full run order. Run `11_simulation.ipynb` first.

Every projection choice so far -- recency weights, an age curve, a replacement baseline -- is a modeling assumption, not a fact. The only way to know whether one choice is actually better than another is to score it against real seasons it never saw. `nuclearff.backtest` is that harness: expanding-window folds, two baseline models to compare against, and the metrics to score them. Mirrors the [Backtesting](https://nolmacdonald.github.io/nuclearff/tutorial/12_backtesting.html) docs page.

## Walk-forward folds

`season_folds` generates expanding-window `(train_seasons, test_season)` pairs -- each fold trains on every season up to a point and tests on the very next one, the way a real projection actually gets used (you never get to peek at the season you're projecting):

In [1]:
from nuclearff.backtest import season_folds

folds = season_folds([2021, 2022, 2023, 2024, 2025], min_train_seasons=2)
folds

[([2021, 2022], 2023),
 ([2021, 2022, 2023], 2024),
 ([2021, 2022, 2023, 2024], 2025)]

## Two baselines

`nuclearff.backtest.walkforward` ships two named baselines to compare a real model against: `baseline_prior_year` (predict this season as an exact copy of last season) and `baseline_recency_weighted` (this tutorial's own recency-weighted blend from `07_projections.ipynb`, applied to whole fantasy-point totals).

## Running a real backtest

`run_walk_forward_backtest` orchestrates fold generation, both models' predictions, and scoring, in one call. Real 2021-2025 WR seasons, scored under this league's own rules:

In [2]:
import polars as pl

from nuclearff.backtest import (
    baseline_prior_year,
    baseline_recency_weighted,
    run_walk_forward_backtest,
)
from nuclearff.config import load_league_config
from nuclearff.config.models import RecencyWeights
from nuclearff.nflverse import load_seasonal_skill_stats
from nuclearff.scoring import ScoringEngine

league_cfg = load_league_config("./demo/configs/leagues/1367225133634191360.yaml")
engine = ScoringEngine(league_cfg.scoring)

seasons = [2021, 2022, 2023, 2024, 2025]
scored = engine.score_frame(load_seasonal_skill_stats(seasons)).filter(
    pl.col("position") == "WR"
)

models = {
    "prior_year": lambda df, test_season: baseline_prior_year(
        df, test_season, "fantasy_points"
    ),
    "recency_weighted": lambda df, test_season: baseline_recency_weighted(
        df, test_season, "fantasy_points", RecencyWeights()
    ),
}
results = run_walk_forward_backtest(
    scored, seasons, "fantasy_points", models, min_train_seasons=2, top_k=12
)
results

ScoringEngine has no mapping for 48 nonzero scoring key(s); these will not be counted toward fantasy points: blk_kick=2.0, blk_kick_ret_yd=0.04, def_3_and_out=0.25, def_4_and_stop=0.5, def_pr_yd=0.04, def_st_ff=1.0, def_st_fum_rec=1.0, def_st_td=6.0, def_td=6.0, ff=1.0, fg_ret_yd=0.04, fgm_0_19=3.0, fgm_20_29=3.0, fgm_30_39=3.0, fgm_40_49=4.0, fgm_50p=5.0, fgmiss=-1.0, fgmiss_0_19=-1.0, fgmiss_20_29=-1.0, fum_rec=2.0, fum_rec_td=6.0, fum_ret_yd=0.04, int=2.0, int_ret_yd=0.04, kr_yd=0.04, pr_yd=0.04, pts_allow_0=5.0, pts_allow_14_20=1.0, pts_allow_1_6=4.0, pts_allow_28_34=-1.0, pts_allow_35p=-4.0, pts_allow_7_13=3.0, sack=1.0, safe=2.0, st_ff=1.0, st_fum_rec=1.0, st_td=6.0, tkl_loss=0.25, xpm=1.0, xpmiss=-1.0, yds_allow_0_100=5.0, yds_allow_100_199=3.0, yds_allow_200_299=2.0, yds_allow_350_399=-1.0, yds_allow_400_449=-3.0, yds_allow_450_499=-5.0, yds_allow_500_549=-6.0, yds_allow_550p=-7.0


model_name,test_season,spearman_correlation,mae,rmse,top_12_precision,top_12_recall,n_players_scored
str,i64,f64,f64,f64,f64,f64,f64
"""prior_year""",2023,0.78849,38.688263,52.433907,0.583333,0.583333,167.0
"""recency_weighted""",2023,0.798623,38.284759,52.069804,0.666667,0.666667,176.0
"""prior_year""",2024,0.745988,48.037803,62.953603,0.416667,0.416667,173.0
"""recency_weighted""",2024,0.752261,46.254719,60.076899,0.5,0.5,186.0
"""prior_year""",2025,0.77587,42.368315,61.57238,0.333333,0.333333,178.0
"""recency_weighted""",2025,0.783989,41.323054,59.028277,0.416667,0.416667,191.0


This is a genuinely useful, real result to look for: across all three real folds, does `recency_weighted` beat the naive `prior_year` baseline on every single metric -- lower error (MAE, RMSE), higher rank correlation, and better top-12 precision/recall? This is exactly the kind of evidence `07_projections.ipynb`'s recency weighting needs before being trusted over the simplest possible alternative -- a claim checked against data, not asserted.

> **Why this matters:** a backtest that only ever confirms your existing model is "working" isn't doing its job. The valuable failure mode to watch for here is a fold where `prior_year` actually *beats* `recency_weighted` -- that's real information about a season where recent form was a worse predictor than simple continuity (an unusually volatile league-wide year, say), not a bug to explain away.

## Scoring one fold by hand

`evaluate_fold` is what `run_walk_forward_backtest` calls once per model per fold -- useful on its own when you already have a prediction frame and just want a score:

In [3]:
from nuclearff.backtest import evaluate_fold

predicted = baseline_recency_weighted(scored, 2025, "fantasy_points", RecencyWeights())
actual = scored.filter(pl.col("season") == 2025).select(["player_id", "fantasy_points"])

metrics = evaluate_fold(
    predicted,
    actual,
    predicted_column="fantasy_points_predicted",
    actual_column="fantasy_points",
    top_k=12,
)
metrics

{'spearman_correlation': 0.7839888448371892,
 'mae': 41.323053851907254,
 'rmse': 59.02827679834992,
 'top_12_precision': 0.4166666666666667,
 'top_12_recall': 0.4166666666666667,
 'n_players_scored': 191.0}

## Individual metrics

Every score above is also available standalone in `nuclearff.backtest.metrics` -- `mae`, `rmse`, `spearman_correlation`, and `top_k_precision_recall` for continuous predictions, plus two more for discrete outputs: `tier_accuracy` (does a predicted draft tier from `08_valuation.ipynb` match the tier the player's real output lands in) and `brier_score` (mean squared error of a predicted probability against a binary real outcome):

In [4]:
from nuclearff.backtest import tier_accuracy
from nuclearff.valuation import assign_tiers

joined = predicted.join(actual, on="player_id", how="inner").drop_nulls()
pred_tiers = assign_tiers(joined["fantasy_points_predicted"].to_list())
actual_tiers = assign_tiers(joined["fantasy_points"].to_list())
tier_accuracy(pred_tiers, actual_tiers)

{'exact_match_rate': 0.3717277486910995, 'within_one_rate': 1.0}

`exact_match_rate` is usually much lower than `within_one_rate` -- the recency-weighted model's predicted tier rarely matches the player's actual-output tier *exactly*, but is close to never off by more than one tier, which is closer to what a draft-day tier board actually needs than exact-tier accuracy alone would suggest.

## What's Next

This chapter validated a *projection* against outcomes it never saw. `13_league_history.ipynb` turns the same stored league data (`04_capturing_a_league.ipynb`) toward a different question -- not "what will happen," but "what already did," across your league's entire real history.

## See Also

- `07_projections.ipynb` -- the recency-weighted model this chapter validates.
- [Backtesting](https://nolmacdonald.github.io/nuclearff/tutorial/12_backtesting.html) -- the matching docs page.